## Code Genration Agent

In [0]:
# Install required packages
%pip install langchain langchain-community databricks-langchain

In [0]:
# Restart Python to use newly installed packages
dbutils.library.restartPython()

In [0]:
# Import necessary libraries
from langchain_core.prompts import PromptTemplate
from databricks_langchain import ChatDatabricks
import os

print("✓ Dependencies imported successfully")

In [0]:
# Configure Databricks Foundation Model API
# No API key needed - authentication is automatic within Databricks!

# Initialize the LLM using Databricks Foundation Model
llm = ChatDatabricks(
    target_uri="databricks",  # Use Databricks serving
    endpoint="databricks-meta-llama-3-3-70b-instruct",  # Free Databricks-hosted model
    temperature=0.2,  # Lower temperature for more consistent code generation
    max_tokens=500
)

print("✓ Databricks Foundation Model configured successfully")
print("Model: Meta Llama 3.3 70B Instruct (free within Databricks)")

In [0]:
# Prompt Engineering: Design a clear and structured prompt template
prompt_template = """
You are an expert code generation assistant. Your task is to generate clean, 
efficient, and well-commented code based on the problem description provided.

Problem Description:
{problem_description}

Programming Language: {language}

Please generate code that:
1. Solves the problem accurately
2. Follows best practices and coding standards
3. Includes helpful comments explaining key logic
4. Is production-ready and efficient

Generated Code:
"""

# Create the PromptTemplate object
code_prompt = PromptTemplate(
    input_variables=["problem_description", "language"],
    template=prompt_template
)

print("✓ Prompt template created")
print("\nInput variables:", code_prompt.input_variables)

In [0]:
# Create a chain combining the prompt template and LLM using LCEL (LangChain Expression Language)
code_generation_chain = code_prompt | llm

print("✓ Code generation chain created successfully")
print(f"\nChain type: {type(code_generation_chain).__name__}")

In [0]:
def generate_code(problem_description: str, language: str = "Python") -> str:
    """
    Code Generation Assistant Function
    
    Args:
        problem_description (str): Description of the problem to solve
        language (str): Programming language for the generated code (default: Python)
    
    Returns:
        str: Generated code snippet
    """
    try:
        # Run the chain with input variables using LCEL invoke method
        result = code_generation_chain.invoke({
            "problem_description": problem_description,
            "language": language
        })
        # LCEL returns an AIMessage object, extract the content
        return result.content
    except Exception as e:
        return f"Error generating code: {str(e)}"

print("✓ Code generation assistant function ready")

In [0]:
# Example 1: Generate code for binary search
problem_1 = "Write a function to implement binary search algorithm that finds the position of a target value in a sorted array."

print("Problem Description:")
print(problem_1)
print("\n" + "="*80 + "\n")

generated_code_1 = generate_code(problem_1, "Python")
print(generated_code_1)

In [0]:
# Example 2: Generate code for data processing
problem_2 = "Create a function that reads a CSV file, filters rows where a specific column value is above a threshold, and returns the result as a pandas DataFrame."

print("Problem Description:")
print(problem_2)
print("\n" + "="*80 + "\n")

generated_code_2 = generate_code(problem_2, "Python")
print(generated_code_2)

In [0]:
# ========================================
# Interactive Code Generator
# ========================================
# Enter your problem in the text box above and click "Run Cell" to generate code!

# Create interactive input widgets
dbutils.widgets.text("problem_input", "", "Enter Your Problem Description")
dbutils.widgets.dropdown("language_input", "Python", ["Python", "JavaScript", "Java", "C++", "Go", "Rust", "TypeScript", "Ruby", "PHP"], "Programming Language")

# Get user input from widgets
user_problem = dbutils.widgets.get("problem_input")
user_language = dbutils.widgets.get("language_input")

# Check if user entered a problem
if not user_problem or user_problem.strip() == "":
    print("⚠️  Please enter a problem description in the text box above and run this cell again.\n")
    print("Example problems you can try:")
    print("  • Create a function that calculates fibonacci numbers recursively")
    print("  • Build a class for managing a shopping cart with add, remove, and total methods")
    print("  • Implement a quicksort algorithm with comments")
    print("  • Create a REST API client with authentication and error handling")
    print("  • Write a function to validate email addresses using regex")
else:
    print("Problem Description:")
    print(user_problem)
    print("\n" + "="*80 + "\n")
    print(f"Generating {user_language} code...\n")
    
    # Generate code for user's problem
    generated_code = generate_code(user_problem, user_language)
    print(generated_code)

## Code Generation Assistant - Summary

### Key LangChain Concepts Demonstrated:

#### 1. **Prompt Engineering**
   - Created a structured prompt template with clear instructions
   - Defined input variables: `problem_description` and `language`
   - Specified expected output format and quality criteria
   - Used lower temperature (0.2) for consistent code generation

#### 2. **Chains (LCEL - LangChain Expression Language)**
   - Combined the prompt template with the LLM using LCEL pipe operator: `code_prompt | llm`
   - Created a reusable `RunnableSequence` that processes inputs and generates outputs
   - Enables sequential processing: Input → Prompt Formatting → LLM → Output
   - Modern approach replacing the deprecated `LLMChain` class

#### 3. **Workflow**
   ```
   Problem Description → Prompt Template → LLM → Generated Code
   ```

### Benefits:
- **Consistency**: Same format for all code generation requests
- **Reusability**: Function can be called multiple times with different inputs
- **Flexibility**: Easy to modify prompt or add new features
- **Scalability**: Can be extended to support multiple languages and complexity levels

### How to Use:
```python
code = generate_code("your problem description", "Python")
print(code)
```

### Implementation Details:
- **LLM Provider**: Databricks Foundation Model API (ChatDatabricks)
- **Model**: Meta Llama 3.3 70B Instruct
- **No API Key Required**: Automatic authentication within Databricks
- **Cost**: Free within Databricks workspace

### Next Steps:
- Add code validation and testing
- Support multiple programming languages
- Add code optimization suggestions
- Integrate with code execution for verification
- Try other Databricks Foundation Models (GPT-OSS, Qwen, Gemma)